# Bibliotecas

In [51]:
import mysql.connector

import undetected_chromedriver as uc

from selenium import webdriver  
from selenium.webdriver.chrome.service import Service 
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

from collections import Counter

import spacy

from fuzzywuzzy import process

import traceback

# Banco De Dados

## Conectando No Banco De Dados

In [52]:
conexao = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = 'BancoDeDado123',
    database = 'projetoUnimar'
)

## Criando o Cursor Para Manipular o Banco De Dados

In [53]:
cursor = conexao.cursor()

# Scraping Do Reclame Aqui!

## Inicializando selenium e webdriver manager

In [54]:
service = Service(ChromeDriverManager().install())  

options = webdriver.ChromeOptions()
#options.add_argument("--headless")

driver = uc.Chrome(options=options)

nlp_ner = spacy.load("Treinar IA/model-last") 

## Coletando os dados do reclame aqui

In [ ]:
produtos_conhecidos = []

def normalizar_produto(produto):
    produto = produto.lower()
    doc = nlp_ner(produto)
    
    lematizado = " ".join([token.lemma_ for token in doc])
    
    if not lematizado.strip():
        lematizado = produto

    if lematizado.endswith('s'):
        lematizado = lematizado.rstrip('s')
    
    resultado = process.extractOne(lematizado, produtos_conhecidos, score_cutoff=80)
    
    if resultado:
        produto_correspondente, similaridade = resultado
        return produto_correspondente
    else:
        produtos_conhecidos.append(lematizado)
    
    return lematizado

empresa = input("Digite a empresa que deseje fazer scraping: ")

quantidade_pagina = int(input("Digite a quantidade de páginas que deseja pegar: "))

produtos_reclamados = []

dicionarioEmpresas = {
    "positivo" : "positivo-informatica",
    "habibs" : "habibs"
}

for pagina in range(quantidade_pagina):
    url = f"https://www.reclameaqui.com.br/empresa/{dicionarioEmpresas.get(empresa)}/lista-reclamacoes/?pagina={pagina + 1}"
    driver.get(url)
    
    try:
        WebDriverWait(driver, 10).until(lambda d: len(d.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')) > 0)
        reclamacoes = driver.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')
        
        for index in range(len(reclamacoes)):
            try:
                reclamacoes = driver.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')
                reclamacao = reclamacoes[index]
                
                driver.execute_script("arguments[0].click();", reclamacao)
                
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'sc-lzlu7c-3')))
                
                titulo = driver.find_element(By.CLASS_NAME, 'sc-lzlu7c-3').text
                reclamacao_texto = driver.find_element(By.CLASS_NAME, 'sc-lzlu7c-17').text.replace('\n', ' ')
                local_reclamacao = driver.find_elements(By.CLASS_NAME, 'sc-lzlu7c-6')[0].text
                data_reclamacao = driver.find_elements(By.CLASS_NAME, 'sc-lzlu7c-6')[1].text
                status_reclamacao = driver.find_element(By.CLASS_NAME, 'sc-lzlu7c-18').text

                produto_identificado = None
                motivo_identificado = None

                doc = nlp_ner(reclamacao_texto)
                for ent in doc.ents:
                    if ent.label_ == "PRODUTO":
                        produto_identificado = ent.text
                        break

                for ent in doc.ents:
                    if ent.label_ == "MOTIVO":
                        motivo_identificado = ent.text
                        break
                
                if produto_identificado:
                    produto_normalizado = normalizar_produto(produto_identificado)
                    produtos_reclamados.append(produto_normalizado)
                
                comando = f'INSERT INTO positivo (titulo_reclamacao, reclamacao, local_reclamacao, data_reclamacao, status_reclamacao, produto_reclamado, motivo_reclamado) VALUES ("{titulo}", "{reclamacao_texto}", "{local_reclamacao}", "{data_reclamacao}", "{status_reclamacao}", "{produto_normalizado}", "{motivo_identificado}")'
                cursor.execute(comando)
                conexao.commit()
                
                driver.get(url)
                WebDriverWait(driver, 10).until(lambda d: len(d.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')) > 0)
            
            except Exception as e:
                print("Erro no processamento de uma reclamação:", e)
                print(traceback.format_exc())
                driver.get(url)
                WebDriverWait(driver, 10).until(lambda d: len(d.find_elements(By.CLASS_NAME, 'sc-1pe7b5t-1')) > 0)

    except Exception as e:
        print("Erro ao acessar a página:", e)
        print(traceback.format_exc())
        driver.get(url)

contador_produtos = Counter(produtos_reclamados)
ranking_produtos = contador_produtos.most_common()

for produto, frequencia in ranking_produtos:
    print(f"{produto}: {frequencia} reclamações")
    comando = f'INSERT INTO positivo (produto_rank, quantidade_rank) VALUES ("`{produto}", "{frequencia}")'

cursor.close()
conexao.close()
driver.quit()

Erro no processamento de uma reclamação: Message: 
Stacktrace:
	GetHandleVerifier [0x00A75093+25075]
	(No symbol) [0x009FE124]
	(No symbol) [0x008DBE63]
	(No symbol) [0x0091FD06]
	(No symbol) [0x0091FF4B]
	(No symbol) [0x0095D8C2]
	(No symbol) [0x00941EC4]
	(No symbol) [0x0095B48E]
	(No symbol) [0x00941C16]
	(No symbol) [0x00913F3C]
	(No symbol) [0x00914ECD]
	GetHandleVerifier [0x00D62523+3094147]
	GetHandleVerifier [0x00D75754+3172532]
	GetHandleVerifier [0x00D6DF32+3141778]
	GetHandleVerifier [0x00B12100+668256]
	(No symbol) [0x00A06C4D]
	(No symbol) [0x00A03DF8]
	(No symbol) [0x00A03F95]
	(No symbol) [0x009F6C80]
	BaseThreadInitThunk [0x76AB5D49+25]
	RtlInitializeExceptionChain [0x77A8CEBB+107]
	RtlGetAppContainerNamedObjectPath [0x77A8CE41+561]

Traceback (most recent call last):
  File "C:\Users\kingu\AppData\Local\Temp\ipykernel_11328\972652286.py", line 53, in <module>
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'sc-lzlu7c-3')))
  File "c:\